<a href="https://colab.research.google.com/github/lahfir/lahfir/blob/main/Story_Writer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install openai-agents pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.1/40.1 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.6/130.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.3/129.3 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 150.9/150.9 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 4.6 MB/s eta 0:00:00


In [ ]:
from agents import Agent, Runner, set_default_openai_key
set_default_openai_key("...")

In [ ]:
from pydantic import BaseModel

class Beat(BaseModel):
    hook: str
    rising_tension: str
    midpoint_shift: str
    dark_turn: str
    climax: str
    resolution: str
    echo_after_effect: str

class StoryOutline(BaseModel):
    logline: str
    beats: Beat

In [ ]:
ideator_agent = Agent(name="Elite Ideator", instructions="You are an elite speculative‑fiction ideator and story consultant.", model="o3", output_type=StoryOutline)

In [ ]:
USER_PROMPT = """Help me generate the seed for a prize‑winning science‑fiction or fantasy short story (≤7,500 words).

Step 1 Brainstorm
• Produce 10 distinct one‑sentence premises.
• Each must be self‑contained, high‑concept, and emotionally intriguing.

Step 2 Score & compare
Create a 10‑row table with columns:
A) Premise, B) Originality (1‑10), C) Thematic depth (1‑10), D) “Sticky” ending potential (1‑10), E) Total score.

Step 3 : Select & expand
• Identify the highest‑scoring premise (break ties by picking the more surprising).
• Write a one‑paragraph *logline* that captures the protagonist, dilemma, setting, and emotional arc.
• Provide a 7‑beat outline using the labels: 1) Hook, 2) Rising tension, 3) Midpoint shift, 4) Dark turn, 5) Climax, 6) Resolution, 7) Echo/after-effect.

Return only the outputs for Steps 1‑3. No commentary."""

import asyncio
story_outline = None
async def main():
  global story_outline
  story_outline = await Runner.run(ideator_agent, USER_PROMPT)
  print(story_outline.final_output)
  story_outline = story_outline.final_output

await main()

logline='Step 1 – Brainstorm\n1) A grief-stricken cartographer on a tidally locked planet discovers the night side isn’t empty but hosts projections of future maps that fade with dawn.\n2) In a feudal society where dragons are bio-engineered war machines, a deaf stable girl realizes she can communicate with them through vibration, threatening the kingdom’s power balance.\n3) A prison ship traveling through a relativistic loop uses inmates’ dreams as fuel; one inmate learns to lucid dream and plots an escape by rewriting reality.\n4) Earth’s ghosts unionize to demand voting rights, possessing living descendants during elections to sway outcomes.\n5) A colony on a sentient fungal network must hold a festival inside its hallucinations every decade to renew symbiosis, but the mushroom starts refusing certain participants.\n6) Time tourists can only visit the moments their ancestors felt most intense emotion; a cynical genealogist catches her future self trying to hijack her heartbreak.\n7)

In [ ]:
class ShortStory(BaseModel):
    story_text: str

In [ ]:
story_agent = Agent(name="Master Storyteller", instructions="You are an award‑winning speculative‑fiction author.", model="o3", output_type=ShortStory)

In [ ]:
story_draft = None

async def generate_story(story_outline: StoryOutline):
  prompt = f"""Using the logline and 7‑beat outline just produced, write a complete short story
• Target length: well under 7,500
• POV: limited third person.
• Voice: lyrical but concise (think Liu Cixin meets Ursula K. Le Guin).
• Show, don’t tell—avoid exposition blocks >150 words.
• End on a decisive yet contemplative line.

Return only the story text, no notes.\n\nLogline: {story_outline.logline}\n\nOutline:\n1) Hook: {story_outline.beats.hook}\n2) Rising tension: {story_outline.beats.rising_tension}\n3) Midpoint shift: {story_outline.beats.midpoint_shift}\n4) Dark turn: {story_outline.beats.dark_turn}\n5) Climax: {story_outline.beats.climax}\n6) Resolution: {story_outline.beats.resolution}\n7) Echo/after-effect: {story_outline.beats.echo_after_effect}"""

  global story_draft
  story_draft = await Runner.run(story_agent, prompt)
  print(story_draft.final_output.story_text)
  story_draft = story_draft.final_output.story_text

async def main():
  await generate_story(story_outline)

await main()

I.

The girl moved through Rivermouth’s Solstice Market like a shadow skimming water. Brass gongs tolled noon, masking the snick of her knife as she lifted purse strings and ribboned lockets from open sleeves. One merchant’s stall glittered brighter than the rest: a velvet tray of vials, each cork wired with silver.

Memories. Fresh-blown, unspooled, still warm.

She palmed one—opal glass, pulse of sunlit gardens inside—then vanished between spice tents. A single heartbeat later the city itself lurched, as though its cobbles had slipped a step down time’s staircase.

Shouts rippled outward. “Who—?” “Where’s—?” Names tangled, then failed. Banners above the gate sagged blank; the bronze equestrian statue at the roundabout stood riderless, saddle empty. Citizens stared at the vacancy with newborn bewilderment.

The girl did not know what had vanished, only that the world suddenly weighed less. She touched the stolen vial. It was colder now, and the gardens within had dimmed to dusk.

II.


In [ ]:
class Scorecard(BaseModel):
  structure_plot_logic: int
  pacing_tension_curve: int
  character_depth_arc: int
  thematic_clarity: int
  emotional_payoff: int
  prose_imagery: int

class EvaluationResult(BaseModel):
  scorecard: Scorecard
  overall_percentage: int
  decision: str
  final_story: str

evaluation_agent = Agent(name="Nebula-winning developmental editor", instructions="You are a Nebula-winning developmental editor and master storyteller.", model="o3", output_type=EvaluationResult)

async def evaluate_story(story_text: str, threshold: int = 85):
  prompt = f"""### **SYSTEM**
You are a Nebula-winning developmental editor and master storyteller.
### **USER**
Evaluate the draft enclosed between <STORY> … </STORY>, then act according to the steps below.
THRESHOLD = {threshold}
─── ① Scorecard (≤20 words per note) ───
Use the table skeleton—fill in scores (1-10) & key notes.
| Category | Score | Key Note |
|----------|-------|----------|
| Structure / Plot logic |   |   |
| Pacing & Tension curve |   |   |
| Character depth & arc  |   |   |
| Thematic clarity       |   |   |
| Emotional payoff       |   |   |
| Prose & imagery        |   |   |
TOTAL = sum of scores (max 60)
OVERALL % = (TOTAL ÷ 60) × 100 (round to whole).
─── ② Decision ───
If OVERALL % ≥ THRESHOLD:
    Output exactly:
DECISION: ACCEPT — no structural rewrite needed.
    Then print the story **unchanged** under the header **FINAL STORY (accepted)**.
Else (OVERALL % < THRESHOLD):
    Output exactly:
DECISION: REVISED — structural rewrite completed.
    Then perform a holistic rewrite that:
    • Fixes every issue noted in the Scorecard.
    • May touch any paragraph for flow; preserve core events, POV, character names, and lyrical tone.
    • Targets 3 300-3 600 words (hard cap 3 400).
    • Keeps the ending’s emotional impact strong.
    Print the new version under the header **FINAL STORY (revised)**.
Return only:
• The filled Scorecard table
• OVERALL %
• DECISION line
• The **FINAL STORY** block (accepted or revised)
<STORY>
{story_text}
</STORY>
"""
  evaluation_result = await Runner.run(evaluation_agent, prompt)
  print(evaluation_result.final_output.overall_percentage)
  return evaluation_result.final_output

refined_story = ""

async def main():
  THRESHOLD = 85
  current_threshold = 0
  global refined_story
  refined_story = story_draft

  while current_threshold < THRESHOLD:
    refined_output = await evaluate_story(refined_story)
    current_threshold = refined_output.overall_percentage
    refined_story = refined_output.final_story

await main()

88


In [ ]:
class PolishedStory(BaseModel):
  text: str

polishing_agent = Agent(name="Master Prose Writer", instructions="You are a master prose stylist blending Octavia E. Butler’s clarity with Ted Chiang’s precision.", model="o3", output_type=PolishedStory)

In [ ]:
USER_PROMPT = f"""Polish the story for rhythm, vivid sensory detail, and emotional punch.

• Cut ~5 % of words by removing redundancies.
• Vary sentence length; keep dialogue snappy.
• Do NOT alter plot or character names.

Return the fully polished story only.

<STORY>
{refined_story}
</STORY>
"""

polished_story = ""

async def main():
  global polished_story
  polished_story = await Runner.run(polishing_agent, USER_PROMPT)
  polished_story = polished_story.final_output.text
  print(polished_story)

await main()

I.

The girl skimmed through Rivermouth’s Solstice Market like a shadow on water. Brass gongs tolled noon, muffling the snick of her knife as she cut purse strings and ribboned lockets from careless sleeves. One stall glittered brightest: a velvet tray of vials, every cork wired with silver.

Memories. Fresh-blown, unspooled, warm.

She palmed one—opal glass, a pulse of sunlit gardens inside—then vanished between spice tents. A single heartbeat later the city lurched, as though its cobbles had slipped a step down time’s staircase.

Shouts rippled outward. “Who—?” “Where’s—?” Names tangled, then failed. Banners above the gate sagged blank; the bronze equestrian statue at the roundabout stood riderless. Citizens stared at the vacancy, newborn bewilderment on every face.

The girl didn’t know what had vanished, only that the world suddenly weighed less. She touched the stolen vial. It was colder now; the gardens within had dimmed to dusk.

II.

By sunset Rivermouth was a clock missing its

In [ ]:
class SubmissionPacket(BaseModel):
    title: str
    author: str = "Generated by AI"
    word_count: int
    story_text: str

submission_agent = Agent(name="Copy Editor", instructions="You are a meticulous copy‑editor and publication formatter.", model="o3", output_type=SubmissionPacket)

In [ ]:
USER_PROMPT = f"""Finalise the submission packet:

1. Correct any lingering grammar, tense, punctuation, or continuity errors.
2. Output in this exact order:

   A) Title
   B) Author: “Generated by AI”
   C) Final word count
   D) The story text

No extra commentary beyond those five items.

<STORY>
{polished_story}
</STORY>
"""

final_story = ""

async def main():
  global final_story
  final_story = await Runner.run(submission_agent, USER_PROMPT)
  final_story = final_story.final_output.story_text
  print(final_story)

await main()

I.

The girl skimmed through Rivermouth’s Solstice Market like a shadow on water. Brass gongs tolled noon, muffling the snick of her knife as she cut purse strings and ribboned lockets from careless sleeves. One stall glittered brightest: a velvet tray of vials, every cork wired with silver.

Memories. Fresh-blown, unspooled, warm.

She palmed one—opal glass, a pulse of sunlit gardens inside—then vanished between spice tents. A single heartbeat later the city lurched, as though its cobbles had slipped a step down time’s staircase.

Shouts rippled outward. “Who—?” “Where’s—?” Names tangled, then failed. Banners above the gate sagged blank; the bronze equestrian statue at the roundabout stood riderless. Citizens stared at the vacancy, newborn bewilderment on every face.

The girl didn’t know what had vanished, only that the world suddenly weighed less. She touched the stolen vial. It was colder now; the gardens within had dimmed to dusk.

II.

By sunset Rivermouth was a clock missing its